# AARB ON vs OFF

`body.update()`

In [25]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

CONFIG_PATH     = ROOT_DIR / 'vehicle_sim' / 'models' / 'params' / 'vehicle_standard.yaml'
AARB_PARAM_PATH = ROOT_DIR / 'vehicle_sim' / 'controllers' / 'anti_roll_bar_controll' / 'param' / 'aarb_param.yaml'
CORNERS = ['FL', 'FR', 'RL', 'RR']

In [26]:
# scenario
from vehicle_sim.scenarios.sinesweep import generate

scenario = generate()

시나리오 생성: dt=0.001s, duration=20.0s, steps=20000, chirp 0.05~1.0 Hz


In [ ]:
from vehicle_sim.models.vehicle_body.vehicle_body import VehicleBody

body = VehicleBody(config_path=str(CONFIG_PATH))

In [ ]:
from vehicle_sim.controllers.anti_roll_bar_controll.active_anti_roll_bar_controller import ActiveAntiRollBarController

aarb = ActiveAntiRollBarController(config_path=str(AARB_PARAM_PATH))

In [ ]:
# simulation
def run(scenario, aarb=None):
    from vehicle_sim.models.vehicle_body.vehicle_body import VehicleBody

    body = VehicleBody(config_path=str(CONFIG_PATH))
    scenario.apply_initial_conditions(body)

    if aarb:
        aarb.reset()

    n = len(scenario['time'])
    log = {k: np.zeros(n) for k in ('roll', 'roll_rate', 'ds_front', 'ds_rear', 'M_arb_f', 'M_arb_r')}

    for i in range(1, n):
        dt = max(scenario['time'][i] - scenario['time'][i - 1], 1e-6)
        idx = i - 1

        T_susp = aarb.update(
            delta_s={c: body.corners[c].suspension.state.delta_s for c in CORNERS},
            delta_s_dot={c: body.corners[c].suspension.state.delta_s_dot for c in CORNERS},
        ) if aarb else {c: 0.0 for c in CORNERS}

        corner_inputs = scenario.corner_inputs(idx=idx, body=body, t_susp=T_susp)
        body.update(dt, corner_inputs)

        log['roll'][i] = body.state.roll
        log['roll_rate'][i] = body.state.roll_rate
        log['ds_front'][i] = body.corners['FL'].suspension.state.delta_s - body.corners['FR'].suspension.state.delta_s
        log['ds_rear'][i] = body.corners['RL'].suspension.state.delta_s - body.corners['RR'].suspension.state.delta_s
        if aarb:
            s = aarb.get_state()
            log['M_arb_f'][i], log['M_arb_r'][i] = s['M_arb_front'], s['M_arb_rear']

    return log

result_off = run(scenario, aarb=None)
result_on = run(scenario, aarb=aarb)

AttributeError: 'dict' object has no attribute 'apply_initial_conditions'

In [ ]:
t   = scenario['time']
off = result_off
on  = result_on

fig, ax = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('AARB ON vs OFF (Drive 50Nm / Steer 짹1Nm 2s)', fontsize=14, fontweight='bold')

for a, y0, y1, yl, tl in [
    (ax[0,0], np.degrees(off['roll']),      np.degrees(on['roll']),      'Roll [deg]',        'Roll Angle'),
    (ax[0,1], np.degrees(off['roll_rate']), np.degrees(on['roll_rate']), 'Roll Rate [deg/s]', 'Roll Rate'),
    (ax[1,0], off['ds_front']*1e3,          on['ds_front']*1e3,          'FL-FR [mm]',        'Front Stroke Diff'),
    (ax[1,1], off['ds_rear']*1e3,           on['ds_rear']*1e3,           'RL-RR [mm]',        'Rear Stroke Diff'),
]:
    a.plot(t, y0, label='OFF', lw=1.5)
    a.plot(t, y1, label='ON',  lw=1.5, ls='--')
    a.set_xlabel('Time [s]'); a.set_ylabel(yl); a.set_title(tl)
    a.legend(); a.grid(alpha=0.3)

for a, key, col, tl in [
    (ax[2,0], 'M_arb_f', 'tab:green',  'Front M_arb (ON only)'),
    (ax[2,1], 'M_arb_r', 'tab:orange', 'Rear M_arb (ON only)'),
]:
    a.plot(t, on[key], color=col, lw=1.5)
    a.set_xlabel('Time [s]'); a.set_ylabel('M [Nm]'); a.set_title(tl)
    a.axhline(0, color='k', ls='--', lw=0.8); a.grid(alpha=0.3)

fig.tight_layout()
plt.show()